[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/limits_course.ipynb)

# Limits and continuity full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and graphs you can move. Work each
question out on paper first, then type your answer into the box and press **check**. It tells you whether you're
right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Fractions as `8/5`, roots as `sqrt(3)`, e as `e` (so e² is `e^2`), π as `pi`. For several
answers, separate them with commas, in the order the question asks.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, cmath, re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x = sp.Symbol("x")
r = sp.Rational
NAMES = {"pi": sp.pi, "e": sp.E, "x": x, "ln": sp.log, "sqrt": sp.sqrt}

def read(s):
    """Your typed answer as exact numbers (a list, one for each answer)."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("π", "pi").replace("²", "^2")
          .replace("³", "^3"))
    s = re.sub(r"√\s*(\d+(?:\.\d+)?|pi|\w)", r"sqrt(\1)", s).replace("√", "sqrt")   # √3 → sqrt(3); √(…) stays
    return [parse_expr(p, local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

def same(a, b):
    try: return abs(complex(sp.N(sp.sympify(a) - sp.sympify(b), 30))) < 1e-9
    except TypeError: return False                         # a letter that isn't e or pi: not a number

def near(f, a, want, d=1e-6, tol=1e-4):
    """f just left and just right of a sits next to the claimed limit."""
    return abs(f(a - d) - want) < tol and abs(f(a + d) - want) < tol

def hole(ax, px, py, color=CORAL, size=11):
    """An open circle: the one point the graph leaves out."""
    ax.plot([px], [py], "o", ms=size, mfc=SLATE, mec=color, mew=2.4, zorder=5)

def table(f, a, steps=(0.1, 0.01, 0.001), digits=5):
    """f near a, from the left and from the right."""
    print(f"{'x':>12} {'f(x)':>14}")
    for h in steps: print(f"{a - h:12.4f} {f(a - h):14.{digits}f}")
    print(f"{'x = ' + str(a):>12} {'never used':>14}")
    for h in reversed(steps): print(f"{a + h:12.4f} {f(a + h):14.{digits}f}")

# Each question: the question, the exact answers in order, the working, and (if it has several parts) the move that
# cracks each part, shown when that part is wrong.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="420px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w) for g, w in zip(got, want)]
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · The idea of a limit

A limit is where a function is **heading** as x gets close to a number. It never needs the value at that number.
Take f(x) = (x² − 1)/(x − 1). At x = 1 it gives 0/0, so a calculator says error. Drag the slider: x walks towards 1
from both sides, and the table fills in.

In [ ]:
f = lambda v: (v * v - 1) / (v - 1)
GAPS = [0.5, 0.2, 0.1, 0.01, 0.001]                 # how far x is from 1, from both sides
assert {v: round(f(v), 3) for v in (0.9, 0.99, 0.999, 1.1, 1.01, 1.001)} == \
       {0.9: 1.9, 0.99: 1.99, 0.999: 1.999, 1.1: 2.1, 1.01: 2.01, 1.001: 2.001}

def approach(step=0):
    fig, (ax, tb) = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={"width_ratios": [1.2, 1]})
    xs = np.linspace(-0.5, 2.5, 301); xs = xs[np.abs(xs - 1) > 1e-9]
    ax.plot(xs, f(xs), color=TEAL, label="f(x) = (x² − 1)/(x − 1)")
    hole(ax, 1, 2)
    for k, g in enumerate(GAPS[:step]):
        last = k == step - 1
        for v in (1 - g, 1 + g):
            ax.plot([v], [f(v)], "o", color=CHALK if last else DIM, ms=10 if last else 6, zorder=6)
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(0, 4); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title("drag the slider" if step == 0 else f"x = 1 ± {GAPS[step - 1]:g}: both sides close in on 2")
    tb.axis("off"); tb.set_xlim(0, 4); tb.set_ylim(len(GAPS) + 1.6, -0.6)
    for c, head in enumerate(("x below 1", "f(x)", "x above 1", "f(x)")):
        tb.text(c + 0.5, 0, head, ha="center", color=DIM, fontsize=13)
    for k, g in enumerate(GAPS[:step], 1):
        for c, val in enumerate((1 - g, f(1 - g), 1 + g, f(1 + g))):
            tb.text(c + 0.5, k, f"{val:g}" if c % 2 == 0 else f"{val:.4g}", ha="center",
                    color=CORAL if c % 2 else CHALK, fontsize=14, family="monospace")
    tb.text(2, len(GAPS) + 1.1, "heading for 2 from both sides" if step >= 3 else "", ha="center", color=CORAL)
    plt.show()

In [ ]:
interact(approach, step=IntSlider(value=0, min=0, max=len(GAPS), step=1, description="closer"));

In [ ]:
Q[1] = {"text": "Without cancelling anything, estimate the limit as x → 2 of (x² + x − 6)/(x − 2). The table "
                "printed above the question comes in from both sides.",
        "answer": [5],
        "working": ["At x = 2 the fraction is 0/0, so look nearby.",
                    "From below: 1.9 gives 4.9, 1.99 gives 4.99, 1.999 gives 4.999.",
                    "From above: 2.1 gives 5.1, 2.01 gives 5.01, 2.001 gives 5.001.",
                    "Both sides head for 5. (The top is (x − 2)(x + 3), so away from 2 the fraction is x + 3.)"]}
g1 = lambda v: (v * v + v - 6) / (v - 2)
table(g1, 2)
assert near(g1, 2, 5)
practice(1)

## 2 · One-sided limits

The left-hand limit comes in from below (x → a⁻), the right-hand limit from above (x → a⁺). The limit exists only
when the two agree.

In [ ]:
Q[2] = {"text": "f(x) = x² + k when x < 2, and f(x) = 3x − 1 when x ≥ 2. With k = 0, give the left-hand limit at 2 "
                "and the right-hand limit at 2. Then give the value of k that makes the limit at 2 exist.",
        "answer": [4, 5, 1],
        "working": ["From the left, f(x) = x² heads for 2² = 4.",
                    "From the right, f(x) = 3x − 1 heads for 3 × 2 − 1 = 5.",
                    "4 ≠ 5, so with k = 0 there is no limit at 2: the graph jumps.",
                    "For the two sides to agree, 4 + k = 5, so k = 1."]}
piece = lambda v, k=0: v * v + k if v < 2 else 3 * v - 1
assert abs(piece(2 - 1e-7) - 4) < 1e-6 and abs(piece(2 + 1e-7) - 5) < 1e-6        # left 4, right 5
assert abs(piece(2 - 1e-7, 1) - piece(2 + 1e-7, 1)) < 1e-6                        # k = 1 closes the gap
practice(2)

## 3 · Limit laws

Limits add, subtract, multiply and divide the way the functions do (as long as you never divide by a zero limit).
So for polynomials and most fractions, the limit is just the value: put the number in.

In [ ]:
Q[3] = {"text": "The limit as x → a of f(x) is 4, and of g(x) it is −3. Find the limit of 2f(x) − g(x)². Then find "
                "the limit as x → 2 of (x³ + 2x)/(x + 2) by direct substitution.",
        "answer": [-1, 3],
        "working": ["Limit laws: 2 × 4 − (−3)² = 8 − 9 = −1.",
                    "The bottom at 2 is 4, not zero, so put 2 in: (8 + 4)/4 = 12/4 = 3."]}
assert 2 * 4 - (-3) ** 2 == -1 and near(lambda v: (v ** 3 + 2 * v) / (v + 2), 2, 3)
practice(3)

## 4 · The indeterminate form 0/0

0/0 isn't an answer: it means a factor is hiding. Three moves crack it. **Factorise** and cancel the shared factor.
**Rationalise**: multiply a square root by its partner. Or use the **standard limit** (xⁿ − aⁿ)/(x − a) → naⁿ⁻¹.
Here is one of each kind. If a part is wrong, the check shows which move cracks it.

In [ ]:
Q[4] = {"text": "Give three limits, in order: (a) x → 4 of (x² − 16)/(x² − 3x − 4); (b) x → 0 of (√(x + 16) − 4)/x; "
                "(c) x → 2 of (x⁴ − 16)/(x − 2).",
        "answer": [r(8, 5), r(1, 8), 32],
        "moves": ["factorise. The top is (x − 4)(x + 4) and the bottom is (x − 4)(x + 1); cancel x − 4.",
                  "rationalise. Multiply top and bottom by √(x + 16) + 4, and the top becomes just x.",
                  "the standard limit (xⁿ − aⁿ)/(x − a) → naⁿ⁻¹, with n = 4 and a = 2 (16 is 2⁴)."],
        "working": ["(a) (x − 4)(x + 4) / ((x − 4)(x + 1)) = (x + 4)/(x + 1) → 8/5.",
                    "(b) × (√(x + 16) + 4) top and bottom: x / (x(√(x + 16) + 4)) = 1/(√(x + 16) + 4) → 1/8.",
                    "(c) 16 = 2⁴, so this is (x⁴ − 2⁴)/(x − 2) → 4 × 2³ = 32."]}
assert near(lambda v: (v * v - 16) / (v * v - 3 * v - 4), 4, 1.6)
assert near(lambda v: (math.sqrt(v + 16) - 4) / v, 0, 0.125)
assert near(lambda v: (v ** 4 - 16) / (v - 2), 2, 32, d=1e-7, tol=1e-3)
practice(4)

## 5 · Limits involving infinity

Near a vertical asymptote the values grow without bound. As x runs off to infinity, divide every term by the
biggest power: the small terms shrink to nothing. A difference of square roots needs its partner first.

In [ ]:
Q[5] = {"text": "Find the limit as x → ∞ of (4x² − x)/(2x² + 5). Then find the limit as x → ∞ of √(x² + 6x) − x.",
        "answer": [2, 3],
        "working": ["Divide top and bottom by x²: (4 − 1/x)/(2 + 5/x²) → 4/2 = 2.",
                    "Multiply by the partner √(x² + 6x) + x, top and bottom: 6x / (√(x² + 6x) + x).",
                    "Divide by x: 6 / (√(1 + 6/x) + 1) → 6/2 = 3."]}
assert abs((4 * 1e8 ** 2 - 1e8) / (2 * 1e8 ** 2 + 5) - 2) < 1e-6
assert abs(6e8 / (math.sqrt(1e16 + 6e8) + 1e8) - 3) < 1e-6
practice(5)

## 6 · The squeeze theorem and trigonometric limits

With x in radians, sin x / x → 1 as x → 0, and (1 − cos x)/x² → ½. Match the angle inside to what you divide by.

In [ ]:
Q[6] = {"text": "Find the limit as x → 0 of sin 5x / (3x). Then find the limit as x → 0 of (1 − cos 6x)/x².",
        "answer": [r(5, 3), 18],
        "working": ["sin 5x / (3x) = (5/3) × sin 5x / (5x) → 5/3 × 1 = 5/3.",
                    "(1 − cos 6x)/x² = 36 × (1 − cos 6x)/(6x)² → 36 × ½ = 18."]}
assert near(lambda v: math.sin(5 * v) / (3 * v), 0, 5 / 3)
assert near(lambda v: (1 - math.cos(6 * v)) / v ** 2, 0, 18, d=1e-3, tol=1e-3)
practice(6)

## 7 · Exponential and logarithmic limits

(1 + 1/n)ⁿ → e as n → ∞. As x → 0, (eˣ − 1)/x → 1 and ln(1 + x)/x → 1.

In [ ]:
Q[7] = {"text": "Find three limits, in order: x → 0 of (e^(4x) − 1)/x; x → 0 of ln(1 + 5x)/(2x); and n → ∞ of "
                "(1 + 2/n)ⁿ.",
        "answer": [4, r(5, 2), sp.E ** 2],
        "working": ["(e^(4x) − 1)/x = 4 × (e^(4x) − 1)/(4x) → 4 × 1 = 4.",
                    "ln(1 + 5x)/(2x) = (5/2) × ln(1 + 5x)/(5x) → 5/2.",
                    "(1 + 2/n)ⁿ = [(1 + 2/n)^(n/2)]², and the inside heads for e, so the limit is e², about 7.389."]}
assert near(lambda v: (math.exp(4 * v) - 1) / v, 0, 4) and near(lambda v: math.log(1 + 5 * v) / (2 * v), 0, 2.5)
assert abs((1 + 2 / 1e7) ** 1e7 - math.e ** 2) < 1e-4
practice(7)

## 8 · Continuity

f is continuous at a when the limit at a exists, f(a) exists, and the two are equal. The intermediate value theorem:
a continuous f that is negative at one end and positive at the other must cross zero in between.

In [ ]:
Q[8] = {"text": "f(x) = kx − 2 when x ≤ 3, and f(x) = x² − 2x when x > 3. Find the k that makes f continuous at 3. "
                "Then, for g(x) = x³ − 2x − 5, give g(2) and g(3): their signs show a root between 2 and 3.",
        "answer": [r(5, 3), -1, 16],
        "working": ["From the right, x² − 2x heads for 9 − 6 = 3. From the left, and at 3, the value is 3k − 2.",
                    "Continuous means 3k − 2 = 3, so k = 5/3.",
                    "g(2) = 8 − 4 − 5 = −1 and g(3) = 27 − 6 − 5 = 16.",
                    "g is a polynomial, so it's continuous: going from −1 to 16, it crosses zero between 2 and 3."]}
assert 3 * r(5, 3) - 2 == 3 ** 2 - 2 * 3 and (2 ** 3 - 2 * 2 - 5, 3 ** 3 - 2 * 3 - 5) == (-1, 16)
practice(8)

## 9 · The precise definition of a limit

lim f(x) = L means: for every band of width ε around L, there is a window of width δ around a where every value
(except at a itself) lands inside the band. For 2x + 1 near 3, δ = ε/2 always works. Shrink ε and watch δ follow.

In [ ]:
def eps_delta(eps=0.5):
    d = eps / 2
    xs = np.linspace(2, 4, 201)
    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    ax.axhspan(7 - eps, 7 + eps, color=CORAL, alpha=0.18, lw=0)
    ax.axvspan(3 - d, 3 + d, color=TEAL, alpha=0.22, lw=0)
    ax.plot(xs, 2 * xs + 1, color=TEAL, label="y = 2x + 1")
    inside = np.linspace(3 - d, 3 + d, 50)
    ax.plot(inside, 2 * inside + 1, color=CORAL, lw=4, label="x in the window: always in the band")
    hole(ax, 3, 7, color=CHALK, size=8)
    ax.set_xlim(2, 4); ax.set_ylim(5, 9); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title(f"band 7 ± {eps:g}  →  window 3 ± {d:g}  (δ = ε/2)")
    plt.show()

In [ ]:
interact(eps_delta, eps=FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05, description="ε"));

In [ ]:
Q[9] = {"text": "The limit as x → 2 of 3x − 1 is 5. For the band ε = 0.06, what is the widest δ that works?",
        "answer": [r(1, 50)],
        "working": ["|(3x − 1) − 5| = |3x − 6| = 3|x − 2|.",
                    "We need 3|x − 2| < 0.06, so |x − 2| < 0.02.",
                    "δ = ε/3 = 0.02 works, and any wider window lets values out of the band."]}
assert all(abs(3 * (2 + t) - 1 - 5) < 0.06 for t in np.linspace(-0.0199, 0.0199, 101))
assert abs(3 * 2.021 - 1 - 5) > 0.06
practice(9)

## 10 · The slope of a curve is a limit

The slope of a chord from a to a + h is (f(a + h) − f(a))/h. Let h shrink to zero, and the chords turn into the
tangent. That limit is the slope of the curve at a.

In [ ]:
Q[10] = {"text": "For y = x² + x, simplify the chord slope from x = 2 to x = 2 + h, and let h shrink. What is the "
                 "slope of the curve at x = 2?",
         "answer": [5],
         "working": ["f(2) = 6, and f(2 + h) = (2 + h)² + (2 + h) = 6 + 5h + h².",
                     "Chord slope: (5h + h²)/h = 5 + h.",
                     "As h → 0 that heads for 5."]}
assert all(abs((((2 + h) ** 2 + 2 + h) - 6) / h - (5 + h)) < 1e-9 for h in (0.5, 0.1, 0.01))
practice(10)

## Sources
NCERT Mathematics Class XI, ch. 12 (Limits and Derivatives); Class XII, ch. 5 (Continuity and Differentiability);
OpenStax, Calculus Volume 1, ch. 2 (Limits). All retrieved 2026-10-31. Every question is our own.